
# RAID (CVPR 2026) → MVTec AD 2 — Full Architecture — AU-PRO@0.05

Notebook này chuyển **official RAID: Retrieval-Augmented Anomaly Detection** sang **MVTec AD 2** theo hướng giữ nguyên kiến trúc và training logic chính của repository, đồng thời thay dataset/evaluation cho đúng bài toán MVTec AD 2.

**Giữ nguyên từ RAID official**
- DINOv2-S/14 frozen feature extractor.
- Hierarchical retrieval: **class prototype → semantic prototype (IVF centroid) → instance tokens**.
- 50 semantic prototypes/class (`nlist=50`), 150 instance neighbors.
- Matching cost volume.
- Guided MoE: 3 experts tầng 1 + 3 experts tầng 2.
- `BinaryFocalLoss + 0.005 × load-balance loss`.
- Adam, LR `1e-4`, 100 epochs.
- Gaussian smoothing `sigma=4`.
- Multi-class/unified training trên toàn bộ category đã chọn.

**Thay đổi cần thiết cho MVTec AD 2**
- Loader đọc archive `data/<category>.tar.gz` theo đúng pattern notebook cũ của bạn, gồm `train`, `validation`, `test_public`, `test_private`, `test_private_mixed`.
- `test_public` được staging thành `test/{good,bad}` chỉ để tương thích code RAID.
- Public metric chính: **AU-PRO@0.05** theo connected-component, mỗi defect region có trọng số bằng nhau.
- Public metric local tính trên anomaly map **256×256** để bám setting input chuẩn 256×256 và giữ RAM khả thi. **Điểm private chính thức** vẫn phải lấy từ MVTec evaluation server với TIFF được resize về **native resolution**.
- FAISS chạy CPU để Colab ổn định hơn; vector index vẫn là `IndexIVFFlat` nên semantics retrieval không đổi.
- Có checkpoint/resume và cache FAISS index trên Google Drive.

> **Quan trọng:** để chạy **toàn bộ kiến trúc RAID**, giữ đủ 8 category. Nếu chỉ để 1 category thì class-level retrieval bị suy biến thành bài toán một lớp và không còn phản ánh đầy đủ RAID.

Official RAID: https://github.com/Mingxiu-Cai/RAID  
MVTec AD 2: https://www.mvtec.com/research-teaching/datasets/mvtec-ad-2


## 1. Mount Drive + cấu hình

In [1]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import os, sys, json, shutil, subprocess, tarfile, random, time, hashlib
from pathlib import PurePosixPath

# ============================================================
# PATHS — bám notebook cũ
# ============================================================
PROJECT_DRIVE = Path(
    "/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9"
)
ARCHIVE_DIR = PROJECT_DRIVE / "data"
OUTPUT_ROOT = PROJECT_DRIVE / "raid_mvtecad2_results"

RAID_REPO = Path("/content/RAID")
RAID_CODE = RAID_REPO / "RAID"
RAID_COMMIT = "165c2c91c4ffee7952c18f97793f561fe171c978"

MAD2_NATIVE_ROOT = RAID_CODE / "data" / "mvtec_ad_2"
RAID_DATA_ROOT = RAID_CODE / "data" / "mvtec_ad_2_raid"

ALL_CATEGORIES = [
    "can",
    "fabric",
    "fruit_jelly",
    "rice",
    "sheet_metal",
    "vial",
    "wallplugs",
    "walnuts",
]

# FULL RAID: giữ đủ 8 category.
CATEGORIES = ALL_CATEGORIES.copy()

# Nếu chỉ smoke-test code:
# CATEGORIES = ["fabric"]

# ============================================================
# RAID / PAPER-LIKE CONFIG
# ============================================================
MODEL_NAME = "dinov2_vits14"
RESOLUTION = 256
DINO_FORWARD_SIZE = 448

EPOCHS = 100
TRAIN_BATCH = 16        # repo default 16; T4 có thể OOM -> giảm 4
NUM_WORKERS = 2
LR = 1e-4
SEED = 1
USE_FAISS_GPU = True

MEMORY_SAMPLES_PER_CLASS = 80
NLIST = 50
NPROBE = 5
KNN_NEIGHBORS = 150

EXPERT_NUM_STAGE1 = 3
EXPERT_NUM_STAGE2 = 3
MOE_BALANCE_WEIGHT = 0.005

MAX_FPR = 0.05
INFERENCE_SEED = 1234

# ============================================================
# FLAGS
# ============================================================
FORCE_EXTRACT = True
FORCE_REBUILD_DB = True
FORCE_TRAIN = False
FORCE_PUBLIC_INFERENCE = False

RUN_PRIVATE_INFERENCE = True
SAVE_PUBLIC_FULLRES_TIFF = True

# RAID's released synthetic-anomaly loader expects DTD at a
# hard-coded path. We satisfy that dependency without changing
# the model/training code.
AUTO_DOWNLOAD_DTD = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert ARCHIVE_DIR.is_dir(), ARCHIVE_DIR
assert len(CATEGORIES) >= 1
assert all(c in ALL_CATEGORIES for c in CATEGORIES)

print("=" * 78)
print("RAID COMMIT       :", RAID_COMMIT)
print("CATEGORIES        :", CATEGORIES)
print("PROJECT_DRIVE     :", PROJECT_DRIVE)
print("ARCHIVE_DIR       :", ARCHIVE_DIR)
print("OUTPUT_ROOT       :", OUTPUT_ROOT)
print("=" * 78)


Mounted at /content/drive
RAID COMMIT       : 165c2c91c4ffee7952c18f97793f561fe171c978
CATEGORIES        : ['can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts']
PROJECT_DRIVE     : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9
ARCHIVE_DIR       : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/data
OUTPUT_ROOT       : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/raid_mvtecad2_results


## 2. Cài dependency + clone đúng RAID commit

In [2]:
import sys
import subprocess
from pathlib import Path

# ============================================================
# 1. CHECK CONFIG FROM PREVIOUS CELL
# ============================================================

assert "RAID_REPO" in globals(), "RAID_REPO chưa được định nghĩa."
assert "RAID_COMMIT" in globals(), "RAID_COMMIT chưa được định nghĩa."

RAID_REPO = Path(RAID_REPO)

print("=" * 80)
print("PYTHON :", sys.version)
print("RAID   :", RAID_REPO)
print("COMMIT :", RAID_COMMIT)
print("=" * 80)


# ============================================================
# 2. REMOVE CPU FAISS
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "faiss-cpu",
        "faiss-gpu-cu12",
    ],
    check=False,
)


# ============================================================
# 3. INSTALL DEPENDENCIES
# ============================================================

packages = [
    "faiss-gpu-cu12==1.14.1.post1",
    "imgaug==0.4.0",
    "einops>=0.7",
    "tifffile>=2024.0",
    "scikit-image>=0.22",
    "scikit-learn>=1.4",
    "scipy>=1.11",
    "pyyaml>=6.0",
]

print("\nInstalling dependencies...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *packages,
    ],
    check=True,
)

print("Dependencies installed.")


# ============================================================
# 4. CLONE / UPDATE RAID
# ============================================================

if not (RAID_REPO / ".git").exists():

    if RAID_REPO.exists() and any(RAID_REPO.iterdir()):
        raise RuntimeError(
            f"{RAID_REPO} đã tồn tại nhưng không phải Git repository."
        )

    print("\nCloning RAID...")

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Mingxiu-Cai/RAID.git",
            str(RAID_REPO),
        ],
        check=True,
    )

else:

    print("\nRAID repository already exists.")
    print("Fetching latest refs...")

    subprocess.run(
        [
            "git",
            "fetch",
            "--all",
            "--tags",
        ],
        cwd=RAID_REPO,
        check=True,
    )


# ============================================================
# 5. PIN EXACT RAID COMMIT
# ============================================================

print("\nChecking out RAID commit...")

subprocess.run(
    [
        "git",
        "checkout",
        "--force",
        "--detach",
        RAID_COMMIT,
    ],
    cwd=RAID_REPO,
    check=True,
)

current = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=RAID_REPO,
    text=True,
).strip()

assert current == RAID_COMMIT, (
    f"Wrong RAID commit\n"
    f"Current : {current}\n"
    f"Expected: {RAID_COMMIT}"
)

print("Pinned RAID:", current)


# ============================================================
# 6. GPU INFO
# ============================================================

print("\n" + "=" * 80)
print("GPU")
print("=" * 80)

subprocess.run(
    ["nvidia-smi"],
    check=False,
)


# ============================================================
# 7. VERIFY FAISS GPU INSTALLATION
# ============================================================

print("\n" + "=" * 80)
print("FAISS INSTALLATION")
print("=" * 80)

# Không import faiss trực tiếp nếu faiss cũ đã được load trong runtime.
if "faiss" in sys.modules:

    print(
        "WARNING: faiss đã được import trước khi đổi package.\n"
        "Hãy Restart Runtime rồi chạy lại notebook từ đầu."
    )

else:

    import faiss

    print("FAISS version :", getattr(faiss, "__version__", "unknown"))

    has_gpu = (
        hasattr(faiss, "StandardGpuResources")
        and hasattr(faiss, "index_cpu_to_gpu")
    )

    print("FAISS GPU     :", has_gpu)

    if not has_gpu:
        raise RuntimeError(
            "FAISS đã cài nhưng không có GPU API. "
            "Hãy Restart Runtime rồi Run all."
        )

    res = faiss.StandardGpuResources()

    cpu_test_index = faiss.IndexFlatL2(384)

    gpu_test_index = faiss.index_cpu_to_gpu(
        res,
        0,
        cpu_test_index,
    )

    print("FAISS GPU test: PASS")


print("\n" + "=" * 80)
print("SETUP COMPLETE")
print("=" * 80)

PYTHON : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
RAID   : /content/RAID
COMMIT : 165c2c91c4ffee7952c18f97793f561fe171c978

Installing dependencies...
Dependencies installed.

Cloning RAID...

Checking out RAID commit...
Pinned RAID: 165c2c91c4ffee7952c18f97793f561fe171c978

GPU

FAISS INSTALLATION
FAISS version : 1.14.1
FAISS GPU     : True
FAISS GPU test: PASS

SETUP COMPLETE


## 3. Giải nén MVTec AD 2 từ các archive per-category

In [3]:

SPLITS_NEEDED = {
    "train",
    "validation",
    "test_public",
    "test_private",
    "test_private_mixed",
}

def split_has_png(root: Path, split: str):
    split_root = root / split
    return split_root.exists() and any(split_root.rglob("*.png"))

def extract_selected_splits(archive_path, destination, splits, force=False):
    destination.mkdir(parents=True, exist_ok=True)

    if not force and all(split_has_png(destination, s) for s in splits):
        print("[SKIP EXTRACT]", destination.name)
        return

    for split in splits:
        shutil.rmtree(destination / split, ignore_errors=True)

    copied = {split: 0 for split in splits}

    with tarfile.open(archive_path, "r:gz") as tf:
        for member in tf.getmembers():
            parts = PurePosixPath(member.name).parts
            positions = [i for i, part in enumerate(parts) if part in splits]
            if not positions:
                continue

            i = positions[0]
            split = parts[i]
            rel_parts = parts[i:]

            if any(part in ("", ".", "..") for part in rel_parts):
                raise RuntimeError(f"Unsafe archive path: {member.name}")

            target = destination.joinpath(*rel_parts)

            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if not member.isfile():
                continue

            target.parent.mkdir(parents=True, exist_ok=True)
            src = tf.extractfile(member)
            if src is None:
                raise RuntimeError(member.name)

            with src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
            copied[split] += 1

    for split in splits:
        assert split_has_png(destination, split), (
            f"Extract lỗi: {destination.name}/{split}"
        )

    print(destination.name, copied)

for category in CATEGORIES:
    archive = ARCHIVE_DIR / f"{category}.tar.gz"
    assert archive.is_file(), f"Không tìm thấy archive: {archive}"

    dst = MAD2_NATIVE_ROOT / category
    extract_selected_splits(
        archive,
        dst,
        SPLITS_NEEDED,
        force=FORCE_EXTRACT,
    )

    audit = {
        "train/good": len(list((dst / "train" / "good").glob("*.png"))),
        "validation/good": len(list((dst / "validation" / "good").glob("*.png"))),
        "test_public/good": len(list((dst / "test_public" / "good").glob("*.png"))),
        "test_public/bad": len(list((dst / "test_public" / "bad").glob("*.png"))),
        "test_public/gt_bad": len(list(
            (dst / "test_public" / "ground_truth" / "bad").glob("*_mask.png")
        )),
        "test_private": len(list((dst / "test_private").rglob("*.png"))),
        "test_private_mixed": len(list((dst / "test_private_mixed").rglob("*.png"))),
    }

    print(category, audit)
    assert all(v > 0 for v in audit.values()), (category, audit)

print("MVTec AD 2 extraction audit: PASS")


can {'test_public': 252, 'test_private': 321, 'validation': 46, 'train': 412, 'test_private_mixed': 321}
can {'train/good': 412, 'validation/good': 46, 'test_public/good': 72, 'test_public/bad': 90, 'test_public/gt_bad': 90, 'test_private': 321, 'test_private_mixed': 321}
fabric {'test_public': 246, 'test_private': 314, 'validation': 43, 'train': 387, 'test_private_mixed': 314}
fabric {'train/good': 387, 'validation/good': 43, 'test_public/good': 66, 'test_public/bad': 90, 'test_public/gt_bad': 90, 'test_private': 314, 'test_private_mixed': 314}
fruit_jelly {'test_public': 140, 'test_private': 255, 'validation': 37, 'train': 263, 'test_private_mixed': 255}
fruit_jelly {'train/good': 263, 'validation/good': 37, 'test_public/good': 20, 'test_public/bad': 60, 'test_public/gt_bad': 60, 'test_private': 255, 'test_private_mixed': 255}
rice {'test_public': 222, 'test_private': 277, 'validation': 35, 'train': 313, 'test_private_mixed': 277}
rice {'train/good': 313, 'validation/good': 35, 'test

## 4. Staging schema cho RAID

In [4]:

def link_dir(src: Path, dst: Path):
    assert src.is_dir(), src

    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)

    dst.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(src, dst, target_is_directory=True)

for category in CATEGORIES:
    staged = RAID_DATA_ROOT / category
    native = MAD2_NATIVE_ROOT / category

    if staged.exists() or staged.is_symlink():
        if staged.is_symlink():
            staged.unlink()
        else:
            shutil.rmtree(staged)

    link_dir(native / "train" / "good", staged / "train" / "good")
    link_dir(native / "test_public" / "good", staged / "test" / "good")
    link_dir(native / "test_public" / "bad", staged / "test" / "bad")
    link_dir(
        native / "test_public" / "ground_truth" / "bad",
        staged / "ground_truth" / "bad",
    )

    assert any((staged / "train" / "good").glob("*.png"))
    assert any((staged / "test" / "good").glob("*.png"))
    assert any((staged / "test" / "bad").glob("*.png"))
    assert any((staged / "ground_truth" / "bad").glob("*_mask.png"))

print("RAID staging:", RAID_DATA_ROOT)
print("STAGING PASS")


RAID staging: /content/RAID/RAID/data/mvtec_ad_2_raid
STAGING PASS



## 5. Chuẩn bị DTD cho synthetic anomaly

RAID released code gọi DTD bằng đường dẫn hard-coded `/home/ZYP/datasets/dtd/images`. Cell này chỉ tạo symlink tới DTD thật; không sửa logic anomaly synthesis của RAID.


In [5]:
import urllib.request
import tarfile
import shutil
import os
from pathlib import Path

drive_dtd_images = ARCHIVE_DIR / "dtd" / "images"
drive_dtd_archive = ARCHIVE_DIR / "dtd-r1.0.1.tar.gz"

local_dtd_root = Path("/content/dtd")
local_dtd_images = local_dtd_root / "images"
local_dtd_archive = Path("/content/dtd-r1.0.1.tar.gz")

DTD_URLS = [
    "https://thor.robots.ox.ac.uk/dtd/dtd-r1.0.1.tar.gz",
    "https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz",
]

if drive_dtd_images.is_dir() and any(drive_dtd_images.rglob("*.jpg")):
    dtd_images = drive_dtd_images

else:
    if not (local_dtd_images.is_dir() and any(local_dtd_images.rglob("*.jpg"))):

        if drive_dtd_archive.is_file() and tarfile.is_tarfile(drive_dtd_archive):
            archive_path = drive_dtd_archive

        else:
            if not AUTO_DOWNLOAD_DTD:
                raise FileNotFoundError(
                    "Thiếu DTD. Đặt dtd-r1.0.1.tar.gz trong ARCHIVE_DIR "
                    "hoặc bật AUTO_DOWNLOAD_DTD."
                )

            archive_path = local_dtd_archive

            if archive_path.exists() and not tarfile.is_tarfile(archive_path):
                print("Removing broken DTD archive:", archive_path)
                archive_path.unlink()

            if not archive_path.exists():
                last_error = None

                for url in DTD_URLS:
                    try:
                        print("Downloading DTD from:")
                        print(url)

                        urllib.request.urlretrieve(url, archive_path)

                        if not tarfile.is_tarfile(archive_path):
                            raise RuntimeError("Downloaded file is not a valid tar.gz")

                        print("DTD download OK.")
                        last_error = None
                        break

                    except Exception as e:
                        last_error = e
                        print("Failed:", repr(e))

                        if archive_path.exists():
                            archive_path.unlink()

                if last_error is not None:
                    raise RuntimeError(
                        "Không download được DTD từ các mirror."
                    ) from last_error

        print("Extracting DTD...")

        with tarfile.open(archive_path, "r:gz") as tf:
            tf.extractall("/content")

    dtd_images = local_dtd_images


assert dtd_images.is_dir(), f"DTD images folder not found: {dtd_images}"
assert any(dtd_images.rglob("*.jpg")), "DTD contains no JPG images"

expected_dtd = Path("/home/ZYP/datasets/dtd/images")
expected_dtd.parent.mkdir(parents=True, exist_ok=True)

if expected_dtd.is_symlink() or expected_dtd.is_file():
    expected_dtd.unlink()
elif expected_dtd.exists():
    shutil.rmtree(expected_dtd)

os.symlink(
    dtd_images,
    expected_dtd,
    target_is_directory=True
)

print("DTD source :", dtd_images)
print("RAID sees  :", expected_dtd)
print("DTD images :", len(list(dtd_images.rglob('*.jpg'))))

https://thor.robots.ox.ac.uk/dtd/dtd-r1.0.1.tar.gz
DTD download OK.
Extracting DTD...


/tmp/ipykernel_977/1480112228.py:73: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall("/content")


DTD source : /content/dtd/images
RAID sees  : /home/ZYP/datasets/dtd/images
DTD images : 5640


## 6. Import RAID + khóa seed + audit kiến trúc

In [6]:

os.chdir(RAID_CODE)
if str(RAID_CODE) not in sys.path:
    sys.path.insert(0, str(RAID_CODE))

# imgaug 0.4 compatibility shim for NumPy 2.x environments.
import numpy as np
if not hasattr(np, "sctypes"):
    np.sctypes = {
        "int": [np.int8, np.int16, np.int32, np.int64],
        "uint": [np.uint8, np.uint16, np.uint32, np.uint64],
        "float": [np.float16, np.float32, np.float64],
        "complex": [np.complex64, np.complex128],
        "others": [bool, object, str, bytes],
    }
for _name, _value in {
    "bool": np.bool_,
    "int": int,
    "float": float,
    "complex": complex,
    "object": object,
    "str": str,
}.items():
    if not hasattr(np, _name):
        setattr(np, _name, _value)

import cv2
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image
from tqdm import tqdm
import faiss
import tifffile
from scipy.ndimage import gaussian_filter

from src.backbones import get_model
from src.moe1 import NonlinearMixtureRes
from src.loss import BinaryFocalLoss
from src.utils import dists2map
from dataset.dataset_multiclass import MVTecDataset

assert torch.cuda.is_available(), "RAID released MoE router assumes CUDA."
DEVICE = "cuda:0"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

HAS_FAISS_GPU = (
    hasattr(faiss, "StandardGpuResources")
    and hasattr(faiss, "index_cpu_to_gpu")
)

print("GPU              :", torch.cuda.get_device_name(0))
print("FAISS GPU support:", HAS_FAISS_GPU)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
faiss.omp_set_num_threads(max(1, min(8, os.cpu_count() or 1)))

CLS = {name: i for i, name in enumerate(CATEGORIES)}
IDX_TO_CLS = {i: name for name, i in CLS.items()}
MASKING = {name: False for name in CATEGORIES}
ROTATION = {name: False for name in CATEGORIES}

print("torch :", torch.__version__)
print("cuda  :", torch.version.cuda)
print("faiss :", getattr(faiss, "__version__", "unknown"))
print("classes:", CLS)


/tmp/ipykernel_977/302882416.py:23: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _name):
/tmp/ipykernel_977/302882416.py:23: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, _name):


GPU              : NVIDIA L4
FAISS GPU support: True
torch : 2.11.0+cu128
cuda  : 12.8
faiss : 1.14.1
classes: {'can': 0, 'fabric': 1, 'fruit_jelly': 2, 'rice': 3, 'sheet_metal': 4, 'vial': 5, 'wallplugs': 6, 'walnuts': 7}


## 7. Load frozen DINOv2-S/14 backbone

In [7]:

backbone = get_model(
    MODEL_NAME,
    DEVICE,
    smaller_edge_size=RESOLUTION,
)
backbone.model.eval()
for p in backbone.model.parameters():
    p.requires_grad_(False)

print("Backbone loaded:", MODEL_NAME)


Loading model: dinov2_vits14
Device: cuda:0
Smaller edge size: 256
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 350MB/s]


Backbone loaded: dinov2_vits14



## 8. Hierarchical vector database

Mỗi category:
1. CLS tokens → 1 class prototype.
2. Patch tokens → `IndexIVFFlat` với `nlist=50` semantic centroids.
3. Toàn bộ patch tokens được giữ trong IVF index làm instance tokens.

Theo released MVTec code của RAID, full-shot memory dùng khoảng 80 normal templates/class và instance retrieval dùng 150 neighbors.


In [8]:

DB_ROOT = OUTPUT_ROOT / "hierarchical_db"
DB_ROOT.mkdir(parents=True, exist_ok=True)

def load_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"Không đọc được ảnh: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

@torch.inference_mode()
def extract_single_repo_style(rgb):
    """
    Bám released RAID:
    prepare_image() -> DINOv2 wrapper transform -> interpolate 448 -> extract.
    """
    image_tensor, grid_size =
￼
arrow_upward
￼
arrow_downward
￼
pen_spark
￼
delete
￼
more_vert
￼
 backbone.prepare_image(rgb)
    image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    image_tensor = F.interpolate(
        image_tensor,
        size=(DINO_FORWARD_SIZE, DINO_FORWARD_SIZE),
        mode="bilinear",
        align_corners=True,
    ).squeeze(0)

    patch, cls_token = backbone.extract_features(image_tensor)
    patch = np.asarray(patch, dtype=np.float32)
    cls_token = np.asarray(cls_token, dtype=np.float32)
    return patch, cls_token, grid_size

def choose_memory_images(category):
    paths = sorted((MAD2_NATIVE_ROOT / category / "train" / "good").glob("*.png"))
    assert paths, category

    n = min(MEMORY_SAMPLES_PER_CLASS, len(paths))
    rng = random.Random(SEED)
    if n == len(paths):
        selected = paths
    else:
        selected = rng.sample(paths, n)
        selected = sorted(selected)
    return selected

def build_or_load_db(category, force=False):
    cat_dir = DB_ROOT / category
    cat_dir.mkdir(parents=True, exist_ok=True)

    index_path = cat_dir / "instance_ivf.faiss"
    proto_path = cat_dir / "class_prototype.npy"
    manifest_path = cat_dir / "memory_images.json"

    if (
        not force
        and index_path.is_file()
        and proto_path.is_file()
        and manifest_path.is_file()
    ):
        index = faiss.read_index(str(index_path))
        index.nprobe = NPROBE
        proto = np.load(proto_path).astype(np.float32)
        print(
            f"[LOAD DB] {category}: ntotal={index.ntotal}, "
            f"nlist={index.nlist}, nprobe={index.nprobe}"
        )
        return proto, index

    selected = choose_memory_images(category)
    patch_features = []
    cls_tokens = []

    for path in tqdm(selected, desc=f"DB/{category}"):
        rgb = load_rgb(path)
        patches, cls_token, grid_size = extract_single_repo_style(rgb)

        mask = backbone.compute_background_mask(
            patches,
            grid_size,
            threshold=10,
            masking_type=MASKING[category],
        )
        patch_features.append(patches[mask])
        cls_tokens.append(cls_token.reshape(1, -1))

    patch_features = np.concatenate(patch_features, axis=0).astype(np.float32)
    cls_tokens = np.concatenate(cls_tokens, axis=0).astype(np.float32)

    # Released RAID uses KMeans(k=1) on CLS tokens.
    cls_kmeans = faiss.Kmeans(
        d=cls_tokens.shape[1],
        k=1,
        niter=100,
        nredo=5,
        verbose=False,
        spherical=False,
        seed=SEED,
        gpu=False,
    )
    cls_kmeans.train(cls_tokens)
    class_proto = cls_kmeans.centroids.astype(np.float32)

    # Normalized L2 == cosine distance relation used by RAID.
    faiss.normalize_L2(patch_features)

    nlist = min(NLIST, max(1, patch_features.shape[0] // 40))
    assert nlist >= 1

    quantizer = faiss.IndexFlatL2(patch_features.shape[1])
    index = faiss.IndexIVFFlat(
        quantizer,
        patch_features.shape[1],
        nlist,
        faiss.METRIC_L2,
    )
    index.train(patch_features)
    index.add(patch_features)
    index.nprobe = min(NPROBE, nlist)

    faiss.write_index(index, str(index_path))
    np.save(proto_path, class_proto)
    manifest_path.write_text(
        json.dumps(
            {
                "category": category,
                "memory_images": [str(p) for p in selected],
                "n_images": len(selected),
                "n_patch_tokens": int(patch_features.shape[0]),
                "feature_dim": int(patch_features.shape[1]),
                "nlist": int(nlist),
                "nprobe": int(index.nprobe),
                "knn_neighbors": KNN_NEIGHBORS,
                "repo_commit": RAID_COMMIT,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"[BUILD DB] {category}: images={len(selected)}, "
        f"tokens={index.ntotal}, nlist={index.nlist}"
    )
    return class_proto, index

CLASS_PROTOTYPES = {}
KNN_INDEX = {}

for category in CATEGORIES:
    proto, index = build_or_load_db(
        category,
        force=FORCE_REBUILD_DB,
    )
    CLASS_PROTOTYPES[category] = proto
    KNN_INDEX[category] = index

SEMANTIC_CENTROIDS = {}

for category, index in KNN_INDEX.items():
    SEMANTIC_CENTROIDS[category] = np.stack(
        [
            index.quantizer.reconstruct(i)
            for i in range(index.nlist)
        ],
        axis=0,
    ).astype(np.float32)


# ============================================================
# SEARCH INDEX
# CPU index vẫn giữ để:
#   - load/save database
#   - semantic centroid search
#
# GPU copy dùng cho:
#   - KNN instance retrieval K=150
# ============================================================

KNN_SEARCH_INDEX = {}

if USE_FAISS_GPU and HAS_FAISS_GPU:

    FAISS_GPU_RES = faiss.StandardGpuResources()

    for category, cpu_index in KNN_INDEX.items():

        gpu_index = faiss.index_cpu_to_gpu(
            FAISS_GPU_RES,
            0,
            cpu_index,
        )

        gpu_index.nprobe = min(
            NPROBE,
            cpu_index.nlist,
        )

        KNN_SEARCH_INDEX[category] = gpu_index

        print(
            f"[FAISS GPU] {category}: "
            f"ntotal={gpu_index.ntotal}, "
            f"nprobe={gpu_index.nprobe}"
        )

else:

    print("[WARN] FAISS GPU unavailable -> CPU fallback")

    for category, cpu_index in KNN_INDEX.items():
        KNN_SEARCH_INDEX[category] = cpu_index


print("Hierarchical DB ready.")


DB/can: 100%|██████████| 80/80 [00:17<00:00,  4.61it/s]


[BUILD DB] can: images=80, tokens=81920, nlist=50


DB/fabric: 100%|██████████| 80/80 [00:28<00:00,  2.80it/s]


[BUILD DB] fabric: images=80, tokens=81920, nlist=50


DB/fruit_jelly: 100%|██████████| 80/80 [00:16<00:00,  4.78it/s]


...

[BUILD DB] vial: images=80, tokens=81920, nlist=50


DB/wallplugs: 100%|██████████| 80/80 [00:17<00:00,  4.55it/s]


[BUILD DB] wallplugs: images=80, tokens=81920, nlist=50


DB/walnuts: 100%|██████████| 80/80 [00:27<00:00,  2.96it/s]


[BUILD DB] walnuts: images=80, tokens=81920, nlist=50
[FAISS GPU] can: ntotal=81920, nprobe=5
[FAISS GPU] fabric: ntotal=81920, nprobe=5
[FAISS GPU] fruit_jelly: ntotal=81920, nprobe=5
[FAISS GPU] rice: ntotal=81920, nprobe=5
[FAISS GPU] sheet_metal: ntotal=81920, nprobe=5
[FAISS GPU] vial: ntotal=81920, nprobe=5
[FAISS GPU] wallplugs: ntotal=81920, nprobe=5
[FAISS GPU] walnuts: ntotal=81920, nprobe=5
Hierarchical DB ready.


## 9. Retrieval + Guided MoE forward

In [9]:

def build_raid_net():
    net = NonlinearMixtureRes(
        EXPERT_NUM_STAGE1,
        EXPERT_NUM_STAGE2,
        768,                 # query 384 + semantic prototype 384
        384,
        KNN_NEIGHBORS,
        1024,                # 32 x 32 DINO patch grid
        "top1",
        strategy="top1",
    ).to(DEVICE)
    return net

def predict_class_from_cls_token(cls_token):
    q = np.asarray(cls_token, dtype=np.float32).reshape(-1)
    q = q / (np.linalg.norm(q) + 1e-12)

    best_name = None
    best_sim = -np.inf
    for category, proto in CLASS_PROTOTYPES.items():
        p = proto.reshape(-1).astype(np.float32)
        p = p / (np.linalg.norm(p) + 1e-12)
        sim = float(np.dot(q, p))
        if sim > best_sim:
            best_sim = sim
            best_name = category

    return best_name, best_sim

def retrieve_for_one(features, category, grid_size=None):
    """
    features: [Q, 384] float32.
    Returns:
      semantic_features [Q,384],
      matching_costs [Q,K],
      mask [Q].
    """
    features = np.asarray(features, dtype=np.float32)
    index = KNN_INDEX[category]

    if grid_size is None:
        mask = np.ones(features.shape[0], dtype=bool)
    else:
        mask = backbone.compute_background_mask(
            features,
            grid_size,
            threshold=10,
            masking_type=MASKING[category],
        )

    query = features[mask].copy()
    faiss.normalize_L2(query)

    # Semantic-level: nearest IVF centroid.
    _, cluster_ids = index.quantizer.search(query, k=1)
    cluster_ids = cluster_ids.reshape(-1)

    semantic = features.copy()
    semantic[mask] = SEMANTIC_CENTROIDS[category][cluster_ids]

    # Instance-level: top-K tokens from probed IVF cells.
    index.nprobe = min(NPROBE, index.nlist)
    distances, _ = index.search(query, k=KNN_NEIGHBORS)

    # Squared L2 on unit vectors: ||a-b||^2 / 2 = 1 - cos(a,b).
    distances = distances.astype(np.float32) / 2.0

    matching = np.full(
        (features.shape[0], KNN_NEIGHBORS),
        0.001,
        dtype=np.float32,
    )
    matching[mask] = distances

    return semantic, matching, mask

def build_moe_inputs_batch(features_batch, categories):
    """
    Batched RAID retrieval.

    features_batch:
        [B, 1024, 384]

    Thay vì FAISS search từng ảnh:
        image 1 -> search
        image 2 -> search
        ...
    ta gom tất cả ảnh cùng category:
        category -> ONE batched FAISS search
    """

    features_batch = np.asarray(
        features_batch,
        dtype=np.float32,
    )

    B, Q, D = features_batch.shape

    assert D == 384, features_batch.shape
    assert Q == 1024, features_batch.shape

    H = W = 32

    semantic_batch = np.empty_like(
        features_batch,
        dtype=np.float32,
    )

    matching_batch = np.empty(
        (B, Q, KNN_NEIGHBORS),
        dtype=np.float32,
    )

    # --------------------------------------------------------
    # Gom sample trong batch theo category
    # --------------------------------------------------------

    category_groups = {}

    for i, category in enumerate(categories):

        if category not in category_groups:
            category_groups[category] = []

        category_groups[category].append(i)

    # --------------------------------------------------------
    # Chỉ search 1 lần / category
    # --------------------------------------------------------

    for category, sample_indices in category_groups.items():

        sample_indices = np.asarray(
            sample_indices,
            dtype=np.int64,
        )

        group_features = features_batch[
            sample_indices
        ]

        n_group = len(sample_indices)

        # [Nc, 1024, 384]
        # ->
        # [Nc*1024, 384]

        query = (
            group_features
            .reshape(-1, D)
            .copy()
        )

        faiss.normalize_L2(query)

        # ====================================================
        # Semantic-level retrieval
        # ====================================================

        cpu_index = KNN_INDEX[category]

        _, cluster_ids = cpu_index.quantizer.search(
            query,
            k=1,
        )

        cluster_ids = cluster_ids.reshape(-1)

        semantic = SEMANTIC_CENTROIDS[
            category
        ][cluster_ids]

        semantic = semantic.reshape(
            n_group,
            Q,
            D,
        )

        semantic_batch[
            sample_indices
        ] = semantic

        # ====================================================
        # Instance-level retrieval
        # GPU FAISS nếu có
        # ====================================================

        search_index = KNN_SEARCH_INDEX[
            category
        ]

        search_index.nprobe = min(
            NPROBE,
            cpu_index.nlist,
        )

        distances, _ = search_index.search(
            query,
            KNN_NEIGHBORS,
        )

        # RAID:
        # ||a-b||² / 2
        # =
        # 1 - cosine(a,b)
        distances = (
            distances.astype(np.float32)
            / 2.0
        )

        distances = distances.reshape(
            n_group,
            Q,
            KNN_NEIGHBORS,
        )

        matching_batch[
            sample_indices
        ] = distances

    # ========================================================
    # NumPy -> GPU Tensor
    # ========================================================

    query_guidance = torch.from_numpy(
        features_batch
    ).to(
        DEVICE,
        dtype=torch.float32,
        non_blocking=True,
    )

    semantic_guidance = torch.from_numpy(
        semantic_batch
    ).to(
        DEVICE,
        dtype=torch.float32,
        non_blocking=True,
    )

    guidance = torch.cat(
        [
            query_guidance,
            semantic_guidance,
        ],
        dim=-1,
    )

    guidance = (
        guidance
        .reshape(B, H, W, -1)
        .permute(0, 3, 1, 2)
        .contiguous()
    )

    matching = torch.from_numpy(
        matching_batch
    ).to(
        DEVICE,
        dtype=torch.float32,
        non_blocking=True,
    )

    matching = (
        matching
        .reshape(
            B,
            H,
            W,
            KNN_NEIGHBORS,
        )
        .permute(0, 3, 1, 2)
        .contiguous()
    )

    return guidance, matching, H, W

print(build_raid_net())


NonlinearMixtureRes(
  (router): Router(
    (conv1): Conv2d(768, 3, kernel_size=(4, 4), stride=(4, 4))
  )
  (router2): Router(
    (conv1): Conv2d(1302, 3, kernel_size=(4, 4), stride=(4, 4))
  )
  (models): ModuleList(
    (0-2): 3 x Expert_layer1(
      (conv): Sequential(
        (0): Conv2d(768, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
  )
  (models2): ModuleList(
    (0-2): 3 x Expert_layer2(
      (proj_x1): Sequential(
        (0): Conv2d(1152, 150, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): GroupNorm(5, 150, eps=1e-05, affine=True)
        (2): SiLU()
      )
      (proj_x2_id): Sequential(
        (0): Conv2d(150, 150, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(5, 150, eps=1e-05, affine=True)
      )
      (q): Linear(in_features=150, out_features=150, bias=False)



## 10. Train Guided MoE trên MVTec AD 2 train/good

Không dùng `test_public` để chọn epoch. Notebook chạy cố định 100 epochs như RAID released config rồi mới đánh giá public.


`validation/good` được giữ riêng đúng protocol MVTec AD 2; vì toàn bộ hyperparameter của notebook này đã khóa theo released RAID nên không dùng `test_public` để tune và cũng không cần dùng validation để chọn epoch.


In [10]:

CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LAST_CKPT = CHECKPOINT_DIR / "raid_mvtecad2_last.pth"
FINAL_CKPT = CHECKPOINT_DIR / "raid_mvtecad2_final.pth"
TRAIN_HISTORY = CHECKPOINT_DIR / "train_history.csv"

def atomic_torch_save(payload, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def train_raid(force=False):
    if FINAL_CKPT.is_file() and not force:
        print("[SKIP TRAIN] Final checkpoint:", FINAL_CKPT)
        return FINAL_CKPT

    train_dataset = MVTecDataset(
        instance_data_root=str(RAID_DATA_ROOT),
        resize=RESOLUTION,
        img_size=RESOLUTION,
        train=True,
        class_name="",
    )
    assert len(train_dataset) > 0

    net = build_raid_net()
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)
    criterion = BinaryFocalLoss().to(DEVICE)

    start_epoch = 0
    history = []

    if TRAIN_HISTORY.is_file() and not force:
        try:
            history = pd.read_csv(TRAIN_HISTORY).to_dict("records")
        except Exception as exc:
            print("[WARN] history:", exc)

    if LAST_CKPT.is_file() and not force:
        state = torch.load(LAST_CKPT, map_location="cpu", weights_only=False)
        net.load_state_dict(state["net"], strict=True)
        optimizer.load_state_dict(state["optimizer"])
        start_epoch = int(state["epoch"]) + 1
        print(f"[RESUME] epoch {start_epoch}/{EPOCHS}")

    print("=" * 78)
    print("TRAIN SAMPLES :", len(train_dataset))
    print("BATCH         :", TRAIN_BATCH)
    print("EPOCHS        :", EPOCHS)
    print("START EPOCH   :", start_epoch)
    print("=" * 78)

    for epoch in range(start_epoch, EPOCHS):
        # Deterministic resume at epoch granularity.
        set_seed(SEED + epoch)

        generator = torch.Generator()
        generator.manual_seed(SEED + epoch)

        # loader = DataLoader(
        #   train_dataset,

        #   batch_size=TRAIN_BATCH,
        #   shuffle=True,

        #   num_workers=NUM_WORKERS,

        #   pin_memory=True,
        #   prefetch_factor=2,

        #   drop_last=False,
        #   generator=generator,
        # )
        # from tqdm import tqdm

        loader = DataLoader(
            train_dataset,
            batch_size=TRAIN_BATCH,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=True,
            prefetch_factor=2,
            drop_last=False,
            generator=generator,
        )

        net.train()
        losses = []
        focal_losses = []
        balance_losses = []

        pbar = tqdm(loader, desc=f"train epoch {epoch+1}/{EPOCHS}")

        for batch in pbar:
            optimizer.zero_grad(set_to_none=True)

            images = batch["anomaly_images"].to(
                DEVICE,
                non_blocking=True,
                dtype=torch.float32,
            )
            gt = batch["anomaly_masks"].to(
                DEVICE,
                non_blocking=True,
                dtype=torch.float32,
            )
            categories = list(batch["class_name"])

            # Released RAID train path: resize normalized synthetic image to 448.
            dino_in = F.interpolate(
                images,
                size=(DINO_FORWARD_SIZE, DINO_FORWARD_SIZE),
                mode="bilinear",
                align_corners=True,
            )

            features, _ = backbone.extract_features(dino_in)
            features = np.asarray(features, dtype=np.float32)

            guidance, matching_logits, H, W = build_moe_inputs_batch(
                features,
                categories,
            )

            outputs, loss_balance, _, _ = net(
                guidance,
                matching_logits,
                epoch,
            )

            outputs = outputs.reshape(-1, 1, H, W)
            outputs = F.interpolate(
                outputs,
                size=(RESOLUTION, RESOLUTION),
                mode="bilinear",
                align_corners=True,
            )

            gt = gt.reshape(-1, 1, RESOLUTION, RESOLUTION)

            loss_focal = criterion(outputs, gt)
            loss = loss_focal + MOE_BALANCE_WEIGHT * loss_balance

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at epoch={epoch}: {loss.item()}"
                )

            loss.backward()
            optimizer.step()

            losses.append(float(loss.detach().cpu()))
            focal_losses.append(float(loss_focal.detach().cpu()))
            balance_losses.append(
                float(loss_balance.detach().cpu())
                if torch.is_tensor(loss_balance)
                else float(loss_balance)
            )

            pbar.set_postfix(
                loss=f"{np.mean(losses[-20:]):.5f}"
            )

        row = {
            "epoch": epoch,
            "loss": float(np.mean(losses)),
            "focal_loss": float(np.mean(focal_losses)),
            "balance_loss": float(np.mean(balance_losses)),
            "lr": float(optimizer.param_groups[0]["lr"]),
        }
        history.append(row)
        pd.DataFrame(history).to_csv(TRAIN_HISTORY, index=False)

        atomic_torch_save(
            {
                "net": net.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
                "config": {
                    "categories": CATEGORIES,
                    "resolution": RESOLUTION,
                    "model_name": MODEL_NAME,
                    "nlist": NLIST,
                    "nprobe": NPROBE,
                    "knn_neighbors": KNN_NEIGHBORS,
                    "repo_commit": RAID_COMMIT,
                },
            },
            LAST_CKPT,
        )

        print(row)

    atomic_torch_save(
        {
            "net": net.state_dict(),
            "epoch": EPOCHS - 1,
            "config": {
                "categories": CATEGORIES,
                "resolution": RESOLUTION,
                "model_name": MODEL_NAME,
                "nlist": NLIST,
                "nprobe": NPROBE,
                "knn_neighbors": KNN_NEIGHBORS,
                "repo_commit": RAID_COMMIT,
            },
        },
        FINAL_CKPT,
    )

    del net, optimizer
    torch.cuda.empty_cache()

    print("[TRAIN COMPLETE]", FINAL_CKPT)
    return FINAL_CKPT

trained_ckpt = train_raid(force=FORCE_TRAIN)


[SKIP TRAIN] Final checkpoint: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/raid_mvtecad2_results/checkpoints/raid_mvtecad2_final.pth


## 11. Load final RAID + inference function

In [11]:

def load_trained_raid():
    assert FINAL_CKPT.is_file(), FINAL_CKPT
    net = build_raid_net()
    state = torch.load(
        FINAL_CKPT,
        map_location="cpu",
        weights_only=False,
    )
    net.load_state_dict(state["net"], strict=True)
    net.eval()
    return net

net = load_trained_raid()

@torch.inference_mode()
def infer_rgb_raid(rgb, deterministic_seed=None):
    """
    Returns:
      amap_256, predicted_category, class_similarity
    """
    patch, cls_token, grid_size = extract_single_repo_style(rgb)

    predicted_category, class_sim = predict_class_from_cls_token(cls_token)

    semantic, matching_np, mask = retrieve_for_one(
        patch,
        predicted_category,
        grid_size=grid_size,
    )

    Q = patch.shape[0]
    H = W = int(round(Q ** 0.5))
    assert H * W == Q == 1024

    guidance_np = np.concatenate([patch, semantic], axis=-1)
    guidance = torch.from_numpy(guidance_np).to(
        DEVICE, dtype=torch.float32
    ).reshape(1, H, W, -1).permute(0, 3, 1, 2)

    matching = torch.from_numpy(matching_np).to(
        DEVICE, dtype=torch.float32
    ).reshape(1, H, W, -1).permute(0, 3, 1, 2)

    if deterministic_seed is not None:
        torch.manual_seed(int(deterministic_seed))
        torch.cuda.manual_seed_all(int(deterministic_seed))

    outputs, _, matching_min, _ = net(
        guidance,
        matching,
        EPOCHS,
    )

    # Released evaluation adds the minimum raw matching residual.
    refined = outputs + matching_min

    amap = F.interpolate(
        refined.reshape(1, 1, H, W),
        size=(RESOLUTION, RESOLUTION),
        mode="bilinear",
        align_corners=True,
    )[0, 0].float().cpu().numpy()

    amap = dists2map(
        amap,
        [RESOLUTION, RESOLUTION],
    ).astype(np.float32)

    if not np.isfinite(amap).all():
        raise RuntimeError("Anomaly map contains NaN/Inf.")

    return amap, predicted_category, class_sim

# Smoke test 1 train image.
test_path = sorted(
    (MAD2_NATIVE_ROOT / CATEGORIES[0] / "train" / "good").glob("*.png")
)[0]
rgb = load_rgb(test_path)
amap, pred_cat, sim = infer_rgb_raid(rgb, INFERENCE_SEED)
print(test_path)
print("map:", amap.shape, amap.min(), amap.max())
print("predicted class:", pred_cat, "sim:", sim)


/content/RAID/RAID/data/mvtec_ad_2/can/train/good/000_regular.png
map: (256, 256) 0.12749414 0.3281247
predicted class: can sim: 0.9250913858413696



## 12. Public inference

Lưu map ở 256×256 để tính AU-PRO@0.05. Có thể bật `SAVE_PUBLIC_FULLRES_TIFF=True` để audit map native-resolution.


In [13]:

PUBLIC_MAP_ROOT = OUTPUT_ROOT / "public_maps_256"
PUBLIC_TIFF_ROOT = OUTPUT_ROOT / "public_maps_fullres"
MANIFEST_ROOT = OUTPUT_ROOT / "manifests"
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

def public_paths(category):
    rows = []
    for kind in ["good", "bad"]:
        for p in sorted(
            (MAD2_NATIVE_ROOT / category / "test_public" / kind).glob("*.png")
        ):
            rows.append((kind, p))
    return rows

def run_public_category(category, force=False):
    manifest_path = MANIFEST_ROOT / f"{category}_test_public.csv"
    expected = public_paths(category)

    if manifest_path.is_file() and not force:
        old = pd.read_csv(manifest_path)
        map_files = [Path(p) for p in old["map_256"].tolist()]
        if len(old) == len(expected) and all(p.is_file() for p in map_files):
            print("[SKIP PUBLIC]", category)
            return old

    rows = []

    for j, (kind, image_path) in enumerate(
        tqdm(expected, desc=f"public/{category}")
    ):
        rgb = load_rgb(image_path)
        h, w = rgb.shape[:2]

        amap_256, pred_cat, class_sim = infer_rgb_raid(
            rgb,
            deterministic_seed=INFERENCE_SEED + j,
        )

        map_path = (
            PUBLIC_MAP_ROOT
            / category
            / kind
            / f"{image_path.stem}.npy"
        )
        map_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(map_path, amap_256.astype(np.float32))

        full_tiff = None
        if SAVE_PUBLIC_FULLRES_TIFF:
            full = cv2.resize(
                amap_256,
                (w, h),
                interpolation=cv2.INTER_LINEAR,
            ).astype(np.float32)

            full_tiff = (
                PUBLIC_TIFF_ROOT
                / category
                / "test_public"
                / kind
                / f"{image_path.stem}.tiff"
            )
            full_tiff.parent.mkdir(parents=True, exist_ok=True)
            tifffile.imwrite(
                full_tiff,
                full.astype(np.float16),
            )

        rows.append({
            "category": category,
            "kind": kind,
            "image": str(image_path),
            "map_256": str(map_path),
            "predicted_class": pred_cat,
            "class_similarity": float(class_sim),
            "fullres_tiff": str(full_tiff) if full_tiff else "",
        })

    df = pd.DataFrame(rows)
    df.to_csv(manifest_path, index=False)
    print("Saved:", manifest_path)
    return df

PUBLIC_MANIFESTS = {}
for category in CATEGORIES:
    PUBLIC_MANIFESTS[category] = run_public_category(
        category,
        force=FORCE_PUBLIC_INFERENCE,
    )

display(pd.concat(PUBLIC_MANIFESTS.values(), ignore_index=True).head())


[SKIP PUBLIC] can
[SKIP PUBLIC] fabric
[SKIP PUBLIC] fruit_jelly
[SKIP PUBLIC] rice
[SKIP PUBLIC] sheet_metal
[SKIP PUBLIC] vial
[SKIP PUBLIC] wallplugs
[SKIP PUBLIC] walnuts


,category,kind,image,map_256,predicted_class,class_similarity,fullres_tiff
0,can,good,/content/RAID/RAID/data/mvtec_ad_2/can/test_pu...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,can,0.937002,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...
1,can,good,/content/RAID/RAID/data/mvtec_ad_2/can/test_pu...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,can,0.942116,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...
2,can,good,/content/RAID/RAID/data/mvtec_ad_2/can/test_pu...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,can,0.938459,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...
3,can,good,/content/RAID/RAID/data/mvtec_ad_2/can/test_pu...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,can,0.924809,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...
4,can,good,/content/RAID/RAID/data/mvtec_ad_2/can/test_pu...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,can,0.924564,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...



## 13. AU-PRO@0.05 — primary metric

Cách tính:
- FPR lấy trên toàn bộ **normal pixels**.
- Mỗi connected component của ground truth là một region.
- PRO là trung bình overlap của các region, vì vậy defect lớn không được ưu tiên hơn defect nhỏ.
- Tích phân PRO–FPR chỉ tới `FPR=0.05`, sau đó chia cho `0.05` để chuẩn hóa về `[0,1]`.


In [15]:

def compute_aupro_005(normal_scores, region_scores, max_fpr=0.05):
    normal_scores = np.concatenate(normal_scores).astype(
        np.float32,
        copy=False,
    )

    region_scores = [
        np.asarray(scores, dtype=np.float32).reshape(-1)
        for scores in region_scores
        if np.asarray(scores).size > 0
    ]

    num_regions = len(region_scores)
    if num_regions == 0:
        raise RuntimeError("Không tìm thấy GT connected component.")
    if normal_scores.size == 0:
        raise RuntimeError("Không có normal pixels để tính FPR.")

    anomaly_scores = np.concatenate(region_scores)

    # Equal weight per connected component, then equal weight per pixel
    # inside each component.
    anomaly_weights = np.concatenate([
        np.full(
            scores.size,
            1.0 / (num_regions * scores.size),
            dtype=np.float64,
        )
        for scores in region_scores
    ])

    scores_all = np.concatenate([normal_scores, anomaly_scores])

    fpr_delta = np.concatenate([
        np.full(
            normal_scores.size,
            1.0 / normal_scores.size,
            dtype=np.float64,
        ),
        np.zeros(anomaly_scores.size, dtype=np.float64),
    ])

    pro_delta = np.concatenate([
        np.zeros(normal_scores.size, dtype=np.float64),
        anomaly_weights,
    ])

    order = np.argsort(scores_all, kind="mergesort")[::-1]
    sorted_scores = scores_all[order]
    cumulative_fpr = np.cumsum(fpr_delta[order])
    cumulative_pro = np.cumsum(pro_delta[order])

    group_ends = np.r_[
        np.flatnonzero(sorted_scores[1:] != sorted_scores[:-1]),
        sorted_scores.size - 1,
    ]

    fpr = np.r_[0.0, cumulative_fpr[group_ends]]
    pro = np.r_[0.0, cumulative_pro[group_ends]]

    unique_fpr, first_idx, counts = np.unique(
        fpr,
        return_index=True,
        return_counts=True,
    )
    unique_pro = pro[first_idx + counts - 1]

    if unique_fpr[-1] < max_fpr:
        raise RuntimeError(f"FPR curve không đạt {max_fpr}.")

    idx = np.searchsorted(unique_fpr, max_fpr, side="left")

    if idx < len(unique_fpr) and np.isclose(unique_fpr[idx], max_fpr):
        x = unique_fpr[:idx + 1]
        y = unique_pro[:idx + 1]
    else:
        x0, x1 = unique_fpr[idx - 1], unique_fpr[idx]
        y0, y1 = unique_pro[idx - 1], unique_pro[idx]
        y_limit = y0 + (max_fpr - x0) * (y1 - y0) / (x1 - x0)

        x = np.r_[unique_fpr[:idx], max_fpr]
        y = np.r_[unique_pro[:idx], y_limit]

    return float(np.trapz(y, x) / max_fpr)

def gt_mask_256(category, kind, image_path):
    if kind == "good":
        return np.zeros((RESOLUTION, RESOLUTION), dtype=np.uint8)

    gt_path = (
        MAD2_NATIVE_ROOT
        / category
        / "test_public"
        / "ground_truth"
        / "bad"
        / f"{image_path.stem}_mask.png"
    )
    assert gt_path.is_file(), gt_path

    gt = np.asarray(Image.open(gt_path).convert("L"))
    gt = (gt > 0).astype(np.uint8)
    gt = cv2.resize(
        gt,
        (RESOLUTION, RESOLUTION),
        interpolation=cv2.INTER_NEAREST,
    )
    return (gt > 0).astype(np.uint8)

def evaluate_public_category(category):
    manifest = PUBLIC_MANIFESTS[category]

    normal_scores = []
    region_scores = []
    n_good = 0
    n_bad = 0

    for row in tqdm(
        manifest.itertuples(index=False),
        total=len(manifest),
        desc=f"AU-PRO0.05/{category}",
    ):
        image_path = Path(row.image)
        kind = row.kind

        amap = np.load(row.map_256).astype(np.float32, copy=False)
        gt = gt_mask_256(category, kind, image_path)

        assert amap.shape == gt.shape == (RESOLUTION, RESOLUTION)

        normal_scores.append(amap[gt == 0].reshape(-1))

        if gt.any():
            n_labels, labels = cv2.connectedComponents(
                gt,
                connectivity=8,
            )
            for region_id in range(1, n_labels):
                region_scores.append(
                    amap[labels == region_id].reshape(-1)
                )

        if kind == "good":
            n_good += 1
        else:
            n_bad += 1

    score = compute_aupro_005(
        normal_scores,
        region_scores,
        max_fpr=MAX_FPR,
    )

    class_acc = float(
        (manifest["predicted_class"] == category).mean()
    )

    return {
        "Category": category,
        "AU_PRO_0.05": score,
        "AU_PRO_0.05_pct": 100.0 * score,
        "public_good": n_good,
        "public_bad": n_bad,
        "gt_regions": len(region_scores),
        "class_retrieval_acc": class_acc,
        "resolution": RESOLUTION,
        "setting": "RAID unified multi-class",
    }

metric_rows = [
    evaluate_public_category(category)
    for category in CATEGORIES
]

metrics_df = pd.DataFrame(metric_rows)

macro = {
    "Category": "MEAN",
    "AU_PRO_0.05": float(metrics_df["AU_PRO_0.05"].mean()),
    "AU_PRO_0.05_pct": float(metrics_df["AU_PRO_0.05_pct"].mean()),
    "public_good": int(metrics_df["public_good"].sum()),
    "public_bad": int(metrics_df["public_bad"].sum()),
    "gt_regions": int(metrics_df["gt_regions"].sum()),
    "class_retrieval_acc": float(metrics_df["class_retrieval_acc"].mean()),
    "resolution": RESOLUTION,
    "setting": "macro mean over selected categories",
}

metrics_df = pd.concat(
    [metrics_df, pd.DataFrame([macro])],
    ignore_index=True,
)

METRICS_PATH = OUTPUT_ROOT / "public_aupro_005.csv"
metrics_df.to_csv(METRICS_PATH, index=False)

display(metrics_df)
print("Saved:", METRICS_PATH)
print(
    f"Macro AU-PRO@0.05 = "
    f"{macro['AU_PRO_0.05_pct']:.4f}%"
)

# Sanity test: perfect separation should give AU-PRO ~= 1.
assert compute_aupro_005(
    [np.array([0.0, 0.0, 0.0], dtype=np.float32)],
    [np.array([1.0, 1.0], dtype=np.float32)],
    max_fpr=0.05,
) > 0.99


AU-PRO0.05/can: 100%|██████████| 162/162 [02:20<00:00,  1.16it/s]
/tmp/ipykernel_977/884218993.py:84: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(y, x) / max_fpr)
AU-PRO0.05/walnuts: 100%|██████████| 150/150 [02:09<00:00,  1.15it/s]


,Category,AU_PRO_0.05,AU_PRO_0.05_pct,public_good,public_bad,gt_regions,class_retrieval_acc,resolution,setting
0,can,0.077952,7.795231,72,90,114,1.0,256,RAID unified multi-class
1,fabric,0.196164,19.616385,66,90,216,1.0,256,RAID unified multi-class
2,fruit_jelly,0.359996,35.999604,20,60,492,1.0,256,RAID unified multi-class
3,rice,0.479516,47.951606,42,90,138,1.0,256,RAID unified multi-class
4,sheet_metal,0.184598,18.459773,24,90,1692,1.0,256,RAID unified multi-class
5,vial,0.659485,65.948534,35,105,156,1.0,256,RAID unified multi-class
6,wallplugs,0.067118,6.711779,60,90,95,1.0,256,RAID unified multi-class
7,walnuts,0.140120,14.011963,60,90,564,1.0,256,RAID unified multi-class
8,MEAN,0.270619,27.061860,379,705,3467,1.0,256,macro mean over selected categories


Saved: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/raid_mvtecad2_results/public_aupro_005.csv
Macro AU-PRO@0.05 = 27.0619%


/tmp/ipykernel_977/884218993.py:84: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(y, x) / max_fpr)



## 14. OPTIONAL — private / private-mixed inference

MVTec AD 2 giữ private GT trên evaluation server. Cell này chỉ sinh **continuous full-resolution TIFF anomaly maps**. Không dùng private data để tune mô hình.


In [20]:

SUBMISSION_ROOT = OUTPUT_ROOT / "submission"

def private_paths(category, split):
    root = MAD2_NATIVE_ROOT / category / split
    paths = sorted(root.rglob("*.png"))
    assert paths, f"Không có ảnh: {root}"
    return paths

def run_private_split(category, split):
    assert split in {"test_private", "test_private_mixed"}

    paths = private_paths(category, split)

    for j, image_path in enumerate(
        tqdm(paths, desc=f"{category}/{split}")
    ):
        out_path = (
            SUBMISSION_ROOT
            / "anomaly_images"
            / category
            / split
            / f"{image_path.stem}.tiff"
        )
        out_path.parent.mkdir(parents=True, exist_ok=True)

        if out_path.is_file():
            continue

        rgb = load_rgb(image_path)
        h, w = rgb.shape[:2]

        amap_256, _, _ = infer_rgb_raid(
            rgb,
            deterministic_seed=INFERENCE_SEED + j,
        )

        full = cv2.resize(
            amap_256,
            (w, h),
            interpolation=cv2.INTER_LINEAR,
        ).astype(np.float32)

        assert np.isfinite(full).all()

        tifffile.imwrite(
            out_path,
            full.astype(np.float16),
        )

    print("DONE:", category, split)

if RUN_PRIVATE_INFERENCE:
    for category in CATEGORIES:
        for split in ["test_private", "test_private_mixed"]:
            run_private_split(category, split)

    archive_path = OUTPUT_ROOT / "raid_mvtecad2_submission.tar.gz"

    import tarfile
    with tarfile.open(archive_path, "w:gz") as tf:
        tf.add(
            SUBMISSION_ROOT / "anomaly_images",
            arcname="anomaly_images",
        )

    print("Submission archive:", archive_path)
else:
    print(
        "SKIP private inference. "
        "Đặt RUN_PRIVATE_INFERENCE=True ở Cell 1 sau khi public metric PASS."
    )


can/test_private: 100%|██████████| 321/321 [02:37<00:00,  2.04it/s]


DONE: can test_private


can/test_private_mixed: 100%|██████████| 321/321 [02:36<00:00,  2.05it/s]


DONE: can test_private_mixed


fabric/test_private: 100%|██████████| 314/314 [03:34<00:00,  1.46it/s]


...

walnuts/test_private: 100%|██████████| 228/228 [00:00<00:00, 779.28it/s]


DONE: walnuts test_private


walnuts/test_private_mixed: 100%|██████████| 228/228 [00:00<00:00, 678.64it/s]


DONE: walnuts test_private_mixed


KeyboardInterrupt: 

## 15. Audit reproducibility

In [17]:

def sha256_file(path, chunk=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

db_stats = {}
for category, index in KNN_INDEX.items():
    db_stats[category] = {
        "ntotal": int(index.ntotal),
        "nlist": int(index.nlist),
        "nprobe": int(index.nprobe),
    }

audit = {
    "raid_repo": "https://github.com/Mingxiu-Cai/RAID",
    "raid_commit": RAID_COMMIT,
    "dataset": "MVTec AD 2",
    "categories": CATEGORIES,
    "model_name": MODEL_NAME,
    "resolution": RESOLUTION,
    "dino_forward_size": DINO_FORWARD_SIZE,
    "epochs": EPOCHS,
    "train_batch": TRAIN_BATCH,
    "lr": LR,
    "memory_samples_per_class": MEMORY_SAMPLES_PER_CLASS,
    "nlist": NLIST,
    "nprobe": NPROBE,
    "knn_neighbors": KNN_NEIGHBORS,
    "expert_num_stage1": EXPERT_NUM_STAGE1,
    "expert_num_stage2": EXPERT_NUM_STAGE2,
    "moe_balance_weight": MOE_BALANCE_WEIGHT,
    "metric": "AU-PRO@0.05",
    "metric_resolution": RESOLUTION,
    "final_checkpoint": str(FINAL_CKPT),
    "final_checkpoint_sha256": sha256_file(FINAL_CKPT),
    "metrics_csv": str(METRICS_PATH),
    "db_stats": db_stats,
    "notes": [
        "Training uses train/good only.",
        "validation/good is reserved for hyperparameters; released RAID settings are fixed here.",
        "test_public is used only after training for local evaluation.",
        "Private splits are inference-only.",
        "FAISS backend is CPU IndexIVFFlat for Colab stability.",
        "Inference router randomness is preserved but deterministically seeded.",
    ],
}

AUDIT_PATH = OUTPUT_ROOT / "audit.json"
AUDIT_PATH.write_text(
    json.dumps(audit, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(audit, indent=2, ensure_ascii=False))
print("AUDIT PASS:", AUDIT_PATH)


{
  "raid_repo": "https://github.com/Mingxiu-Cai/RAID",
  "raid_commit": "165c2c91c4ffee7952c18f97793f561fe171c978",
  "dataset": "MVTec AD 2",
  "categories": [
    "can",
    "fabric",
    "fruit_jelly",
    "rice",
    "sheet_metal",
    "vial",
    "wallplugs",
    "walnuts"
  ],
  "model_name": "dinov2_vits14",
  "resolution": 256,
  "dino_forward_size": 448,
  "epochs": 100,
  "train_batch": 16,
  "lr": 0.0001,
  "memory_samples_per_class": 80,
  "nlist": 50,
  "nprobe": 5,
  "knn_neighbors": 150,
  "expert_num_stage1": 3,
  "expert_num_stage2": 3,
  "moe_balance_weight": 0.005,
  "metric": "AU-PRO@0.05",
  "metric_resolution": 256,
  "final_checkpoint": "/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/raid_mvtecad2_results/checkpoints/raid_mvtecad2_final.pth",
  "final_checkpoint_sha256": "3850600c73160f992dcd899a1a693fee42e559ff4aeb5a4e744226c6b76a93ac",
  "metrics_csv": "/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/raid_mvtecad2_results


## Kết quả cần lấy

Sau khi chạy đủ notebook:

- **Primary**: `raid_mvtecad2_results/public_aupro_005.csv`
- Checkpoint: `raid_mvtecad2_results/checkpoints/raid_mvtecad2_final.pth`
- Hierarchical DB: `raid_mvtecad2_results/hierarchical_db/`
- Public maps: `raid_mvtecad2_results/public_maps_256/`
- Audit: `raid_mvtecad2_results/audit.json`
- Private submission (khi bật): `raid_mvtecad2_results/raid_mvtecad2_submission.tar.gz`

### Diễn giải khoa học
RAID gốc chưa công bố benchmark MVTec AD 2 trong repository này. Vì vậy số AU-PRO@0.05 thu được ở đây là **kết quả adaptation RAID → MVTec AD 2**, không phải con số reproduction của bảng gốc RAID trên MVTec-AD.

Để so sánh nghiên cứu công bằng, giữ:
- cùng 8 category,
- cùng `RESOLUTION=256`,
- không tune bằng `test_public`,
- cùng checkpoint cuối,
- báo cáo per-category + macro mean AU-PRO@0.05,
- và dùng private evaluation server làm kết quả chính thức nếu viết paper.
